
# Tiny Tao: Population Neural Networks + Locked Certificates

This notebook is `tiny_tao_osc_gpu.ipynb` plus the missing close-the-loop step.

Geometry (phase MAE, \(\gcd(k,n)\), radius \(q\)) is a ranking, not a certificate.
A net is **Family F** or **Family Q** only if it is domain-exact in float32
(`domain_acc >= 1`) **and** a neural-free decoder reconstructs every pair of
\(\mathbb F_p^\times\) multiplication. Population ids are the checkpoint ids.

Same training as the original: Model C, \(H=1\), shared 2-D embedding,
complex multiply, per-net linear readout, held-row or full table, vectorized
population on one GPU.

It supports:

- full-table or held-row training;
- thousands of independently initialized networks in parallel;
- exact full-domain solver counts;
- GPU utilization / memory diagnostics;
- faithful-character geometry;
- kernel-2 / quotient+radial geometry;
- **Family F** roots-of-unity decoder (100% required);
- **Family Q** sector+shell decoder (100% required; \(p\equiv 3\pmod 4\));
- composition nearest-neighbor decoder and character-into-learned-\(W\) substitution;
- saved checkpoints, bound net ids, and certificate CSVs.

Self-contained for **OSC OnDemand Jupyter** with one GPU. Does not import the lab repo.



## OSC launch notes

In OSC OnDemand, start a **Jupyter** interactive session and request a GPU in the launch form.
OSC's current Jupyter interface automatically selects an appropriate node type from the requested
compute resources.

For this experiment, start with:

- **1 GPU**
- **4–8 CPU cores**
- **16–32 GB system RAM**
- **2 hours wall time** for experimentation
- one ordinary Jupyter session, not Jupyter+LLM

Cardinal H100 or Ascend A100 hardware is ideal, but the notebook only assumes CUDA-capable PyTorch.

The first code cell verifies that the notebook actually landed on a GPU compute node.


In [ ]:

import os, sys, time, math, json, platform, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Stop here and relaunch the OSC Jupyter session with a GPU request."
    )

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime seen by PyTorch:", torch.version.cuda)
print("GPU capability:", torch.cuda.get_device_capability(0))
print("Allocated now (GB):", torch.cuda.memory_allocated(0)/1e9)
print("Reserved now (GB):", torch.cuda.memory_reserved(0)/1e9)

print("\n--- nvidia-smi ---")
subprocess.run(
    ["nvidia-smi",
     "--query-gpu=name,utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu",
     "--format=csv,noheader,nounits"],
    check=False
)


## Experiment configuration

In [ ]:

# ---- Core settings ----
# Defaults match tiny_tao_osc_gpu.ipynb. For the pasted p=11 POP=4096
# geometry table, set P=11 and POP=4096. Certificates do not change training.
P = 13                   # prime modulus
POP = 1024               # number of independently initialized tiny networks
STEPS = 8000
LR = 3e-3
SEED = 20260909

# Split:
# "full"     -> train all pairs
# "held_row" -> hold a = P-1 out of training, while it remains present as b
SPLIT = "held_row"
HELD_A = P - 1

# Check exactness every this many steps.
CHECK_EVERY = 250

# Save output here.
RUN_DIR = Path.cwd() / "tiny_tao_results" / f"p{P}_{SPLIT}_{int(time.time())}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

print("Run directory:", RUN_DIR)



## Dataset

We work on the nonzero residues \(1,\ldots,p-1\). For prime \(p\), these form the cyclic group
\(\mathbb F_p^\times\).

For the held-row experiment, \(a=p-1\) is omitted **only from the first operand slot during
training**. Because the embedding is shared between the two operand slots, that residue is still
trained whenever it appears as \(b\). All output classes remain visible in the training set.


In [ ]:

def make_table(p):
    elems = torch.arange(1, p, dtype=torch.long)
    a, b = torch.meshgrid(elems, elems, indexing="ij")
    y = (a * b) % p

    # Convert displayed residue labels 1..p-1 into class indices 0..p-2.
    # For nonzero products modulo prime p, y is never 0.
    y_class = y - 1
    return elems, a.reshape(-1), b.reshape(-1), y_class.reshape(-1)

elems, a_all_cpu, b_all_cpu, y_all_cpu = make_table(P)

if SPLIT == "full":
    train_mask_cpu = torch.ones_like(a_all_cpu, dtype=torch.bool)
elif SPLIT == "held_row":
    train_mask_cpu = a_all_cpu != HELD_A
else:
    raise ValueError(SPLIT)

test_mask_cpu = ~train_mask_cpu

# Convert residue values 1..p-1 to embedding indices 0..p-2.
a_idx = (a_all_cpu - 1).to(device)
b_idx = (b_all_cpu - 1).to(device)
y_all = y_all_cpu.to(device)
train_mask = train_mask_cpu.to(device)
test_mask = test_mask_cpu.to(device)

print("Group order:", P-1)
print("Full pairs:", len(y_all_cpu))
print("Train pairs:", int(train_mask_cpu.sum()))
print("Held-out pairs:", int(test_mask_cpu.sum()))

train_classes = set(y_all_cpu[train_mask_cpu].tolist())
all_classes = set(range(P-1))
print("All output classes present in train:", train_classes == all_classes)



## Model C, vectorized across the population

For each independent network \(r\), learn one shared complex embedding for every residue,

\[
z_r(a)=x_r(a)+i\,y_r(a).
\]

Composition is fixed complex multiplication:

\[
h_r(a,b)=z_r(a)z_r(b).
\]

A network-specific linear readout maps the 2-D product to the \(p-1\) classes.

Every network has independent parameters, but all networks are trained simultaneously in one
set of GPU tensor operations.


In [ ]:

N = P - 1

# E[r, a, xy]
E = torch.empty(POP, N, 2, device=device, requires_grad=True)

# W[r, xy, class]
W = torch.empty(POP, 2, N, device=device, requires_grad=True)

# b[r, class]
bias = torch.zeros(POP, N, device=device, requires_grad=True)

with torch.no_grad():
    # Xavier-like scales
    E.normal_(mean=0.0, std=math.sqrt(2.0 / (N + 2)))
    W.normal_(mean=0.0, std=math.sqrt(2.0 / (2 + N)))

optimizer = torch.optim.Adam([E, W, bias], lr=LR)

param_per_net = N*2 + 2*N + N
print(f"Parameters/network: {param_per_net}")
print(f"Population parameters: {POP * param_per_net:,}")


In [ ]:

def forward_all(E, W, bias):
    # embeddings for every ordered pair
    ea = E[:, a_idx, :]   # [POP, pairs, 2]
    eb = E[:, b_idx, :]   # [POP, pairs, 2]

    xa, ya = ea[..., 0], ea[..., 1]
    xb, yb = eb[..., 0], eb[..., 1]

    # complex multiplication
    h = torch.stack(
        [xa*xb - ya*yb,
         xa*yb + ya*xb],
        dim=-1
    )                    # [POP, pairs, 2]

    logits = torch.einsum("rpi,ric->rpc", h, W) + bias[:, None, :]
    return logits, h

@torch.no_grad()
def accuracies(E, W, bias):
    logits, _ = forward_all(E, W, bias)
    pred = logits.argmax(dim=-1)
    target = y_all[None, :]

    correct = pred.eq(target)

    train_acc = correct[:, train_mask].float().mean(dim=1)
    test_acc = (
        correct[:, test_mask].float().mean(dim=1)
        if int(test_mask.sum()) > 0
        else torch.ones(POP, device=device)
    )
    domain_acc = correct.float().mean(dim=1)
    exact = correct.all(dim=1)

    return train_acc, test_acc, domain_acc, exact


## GPU throughput benchmark

In [ ]:

# Short warm-up + benchmark of the exact workload.
BENCH_STEPS = 100

for _ in range(10):
    optimizer.zero_grad(set_to_none=True)
    logits, _ = forward_all(E, W, bias)
    logits_train = logits[:, train_mask, :]
    targets = y_all[train_mask][None, :].expand(POP, -1)

    loss = F.cross_entropy(
        logits_train.reshape(-1, N),
        targets.reshape(-1),
        reduction="mean"
    )
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()
t0 = time.perf_counter()

for _ in range(BENCH_STEPS):
    optimizer.zero_grad(set_to_none=True)
    logits, _ = forward_all(E, W, bias)
    logits_train = logits[:, train_mask, :]
    targets = y_all[train_mask][None, :].expand(POP, -1)

    loss = F.cross_entropy(
        logits_train.reshape(-1, N),
        targets.reshape(-1),
        reduction="mean"
    )
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

print(f"{BENCH_STEPS} steps: {elapsed:.3f} s")
print(f"steps/s: {BENCH_STEPS/elapsed:.2f}")
print(f"network-steps/s: {POP*BENCH_STEPS/elapsed:,.0f}")
print(f"estimated {STEPS} steps: {STEPS/(BENCH_STEPS/elapsed)/60:.1f} min")

subprocess.run(
    ["nvidia-smi",
     "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu",
     "--format=csv,noheader"],
    check=False
)



## Train the population

The loss is averaged across every network and every visible pair, but because each network has
its own parameters there is no information sharing between population members.

Exactness is evaluated using integer class predictions over the complete domain.


In [ ]:

history = []
torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()

for step in range(1, STEPS + 1):
    optimizer.zero_grad(set_to_none=True)

    logits, _ = forward_all(E, W, bias)
    logits_train = logits[:, train_mask, :]
    targets = y_all[train_mask][None, :].expand(POP, -1)

    loss = F.cross_entropy(
        logits_train.reshape(-1, N),
        targets.reshape(-1),
        reduction="mean"
    )

    loss.backward()
    optimizer.step()

    if step == 1 or step % CHECK_EVERY == 0 or step == STEPS:
        train_acc, test_acc, domain_acc, exact = accuracies(E, W, bias)
        exact_n = int(exact.sum())

        rec = {
            "step": step,
            "loss": float(loss.detach()),
            "mean_train_acc": float(train_acc.mean()),
            "mean_test_acc": float(test_acc.mean()),
            "mean_domain_acc": float(domain_acc.mean()),
            "exact": exact_n,
        }
        history.append(rec)

        print(
            f"{step:5d}  loss={rec['loss']:.6f}  "
            f"train={rec['mean_train_acc']:.4f}  "
            f"test={rec['mean_test_acc']:.4f}  "
            f"domain={rec['mean_domain_acc']:.4f}  "
            f"exact={exact_n}/{POP}"
        )

torch.cuda.synchronize()
train_seconds = time.perf_counter() - t0
print(f"\nTraining time: {train_seconds/60:.2f} min")
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


## Save metrics and successful networks

In [ ]:

train_acc, test_acc, domain_acc, exact = accuracies(E, W, bias)
exact_ids = torch.where(exact)[0]

pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)

summary = {
    "p": P,
    "population": POP,
    "steps": STEPS,
    "lr": LR,
    "split": SPLIT,
    "held_a": HELD_A if SPLIT == "held_row" else None,
    "seed": SEED,
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "train_seconds": train_seconds,
    "exact_count": int(exact.sum()),
    "mean_train_accuracy": float(train_acc.mean()),
    "mean_test_accuracy": float(test_acc.mean()),
    "mean_domain_accuracy": float(domain_acc.mean()),
    "peak_gpu_memory_gb": torch.cuda.max_memory_allocated()/1e9,
}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

# Bind every population slot, not only the exact subset. Geometry tables
# must use these ids; local 0..n_exact-1 indices are not checkpoint ids.
torch.save(
    {
        "E": E.detach().cpu(),
        "W": W.detach().cpu(),
        "bias": bias.detach().cpu(),
        "domain_acc": domain_acc.detach().cpu(),
        "exact": exact.detach().cpu(),
        "p": P,
        "split": SPLIT,
        "seed": SEED,
        "population": POP,
    },
    RUN_DIR / "population.pt",
)

if len(exact_ids):
    torch.save(
        {
            "network_ids": exact_ids.cpu(),
            "E": E.detach()[exact_ids].cpu(),
            "W": W.detach()[exact_ids].cpu(),
            "bias": bias.detach()[exact_ids].cpu(),
            "p": P,
            "split": SPLIT,
        },
        RUN_DIR / "successful.pt",
    )

print("exact ids (checkpoint / population slots):", exact_ids.cpu().tolist()[:32],
      "..." if len(exact_ids) > 32 else "")
summary



# Mechanistic analysis

The following cells look for the two solver families seen in the Tiny Tao experiments.

For prime \(p\), choose a primitive root \(g\) and assign each residue its discrete-log exponent
\(j\), so \(a=g^j\bmod p\).

A **faithful** one-circle solution has phase approximately

\[
\theta(a)=\phi+\frac{2\pi k j}{p-1},
\qquad \gcd(k,p-1)=1.
\]

A possible **kernel-2 / quotient+radial** solution has \(\gcd(k,p-1)=2\), with phase collapsing
\(a\) and \(-a\) and radius potentially carrying the missing binary coordinate.


In [ ]:

def prime_factors(n):
    out = set()
    d = 2
    while d*d <= n:
        while n % d == 0:
            out.add(d)
            n //= d
        d += 1
    if n > 1:
        out.add(n)
    return out

def primitive_root_prime(p):
    phi = p - 1
    fac = prime_factors(phi)
    for g in range(2, p):
        if all(pow(g, phi//q, p) != 1 for q in fac):
            return g
    raise RuntimeError("No primitive root found")

def discrete_log_table(p, g):
    table = {}
    x = 1
    for j in range(p-1):
        table[x] = j
        x = (x*g) % p
    return table

g = primitive_root_prime(P)
dlog = discrete_log_table(P, g)
j = torch.tensor([dlog[a] for a in range(1, P)], dtype=torch.float64)

print("Primitive root:", g)
print("Discrete-log order:", [pow(g, k, P) for k in range(P-1)])


In [ ]:

def wrap_angle(x):
    return torch.atan2(torch.sin(x), torch.cos(x))

def fit_windings(E_subset, p, j, net_ids):
    # E_subset: [R, N, 2] on CPU. net_ids are population / checkpoint slots.
    z = E_subset[..., 0].double() + 1j * E_subset[..., 1].double()
    theta = torch.angle(z)
    radius = torch.abs(z)

    rows = []
    n = p - 1
    net_ids = [int(x) for x in net_ids]

    for r in range(E_subset.shape[0]):
        best = None
        for k in range(n):
            ideal = 2*math.pi*k*j/n
            # circular mean of theta - ideal gives global phase
            delta = theta[r] - ideal
            phi = torch.angle(torch.exp(1j*delta).mean())
            resid = wrap_angle(theta[r] - ideal - phi)
            mae_deg = float(resid.abs().mean() * 180/math.pi)

            candidate = (mae_deg, k, float(phi))
            if best is None or candidate < best:
                best = candidate

        mae_deg, k, phi = best
        rad = radius[r]
        rows.append({
            "net": net_ids[r],
            "local_idx": r,
            "best_k": k,
            "gcd_k_n": math.gcd(k, n),
            "phase_mae_deg": mae_deg,
            "radius_cv": float(rad.std() / rad.mean()),
            "mean_radius": float(rad.mean()),
            "phi": phi,
        })

    return pd.DataFrame(rows)

if len(exact_ids):
    E_success = E.detach()[exact_ids].cpu()
    W_success = W.detach()[exact_ids].cpu()
    bias_success = bias.detach()[exact_ids].cpu()
    mech = fit_windings(E_success, P, j, exact_ids.cpu().tolist())
    display(mech.head(20))
    print("\nGCD counts:")
    print(mech["gcd_k_n"].value_counts().sort_index())
else:
    print("No exact networks to analyze.")



## Direct homomorphism diagnostic

After removing the global phase gauge, compare

\[
\theta(ab)
\]

with

\[
\theta(a)+\theta(b).
\]

For a true complex character, the centered residual should be close to zero over the entire table.


In [ ]:

@torch.no_grad()
def homomorphism_errors(E_subset, p, net_ids):
    # CPU analysis. net_ids are population / checkpoint slots.
    z = E_subset[..., 0].double() + 1j*E_subset[..., 1].double()
    theta = torch.angle(z)

    aa = a_all_cpu - 1
    bb = b_all_cpu - 1
    cc = y_all_cpu  # class index = product residue - 1

    out = []
    for r in range(E_subset.shape[0]):
        delta = wrap_angle(theta[r, cc] - theta[r, aa] - theta[r, bb])
        gauge = torch.angle(torch.exp(1j*delta).mean())
        centered = wrap_angle(delta - gauge)
        out.append({
            "net": int(net_ids[r]),
            "mean_abs_deg": float(centered.abs().mean()*180/math.pi),
            "max_abs_deg": float(centered.abs().max()*180/math.pi),
            "gauge_deg": float(gauge*180/math.pi),
        })
    return pd.DataFrame(out)

if len(exact_ids):
    hom = homomorphism_errors(E_success, P, exact_ids.cpu().tolist())
    mech2 = mech.merge(hom, on="net")
    display(mech2.sort_values("mean_abs_deg").head(20))
    mech2.to_csv(RUN_DIR / "mechanistic_summary.csv", index=False)



## Kernel-2 radial diagnostic

For a kernel-2 winding, the angular representation collapses two elements in each fiber.
For primes \(p\equiv3\pmod4\), the p=11-style alternative solver may encode the missing
\(C_2\) coordinate in the radius.

This cell checks whether radius separates even and odd discrete-log parity.


In [ ]:

if len(exact_ids):
    zsucc = E_success[..., 0].double() + 1j*E_success[..., 1].double()
    rsucc = torch.abs(zsucc)

    parity = (j.long() % 2)
    even_mask = parity == 0
    odd_mask = parity == 1

    rows = []
    for r in range(len(exact_ids)):
        re = float(rsucc[r, even_mask].mean())
        ro = float(rsucc[r, odd_mask].mean())
        rows.append({
            "net": int(exact_ids[r]),
            "rho_even": re,
            "rho_odd": ro,
            "q_odd_over_even": ro/re if re != 0 else np.nan,
            "radius_parity_separation":
                abs(re-ro) / ((re+ro)/2) if (re+ro) else np.nan,
        })

    radial = pd.DataFrame(rows)
    mech3 = mech2.merge(radial, on="net")
    display(
        mech3.sort_values(
            ["gcd_k_n", "phase_mae_deg"]
        ).head(30)
    )
    mech3.to_csv(RUN_DIR / "mechanistic_with_radius.csv", index=False)


## Close the loop

The tables above rank **geometry**. They do not replace a network.

A certificate requires all of:

1. Bind `net` to a population / checkpoint slot (already done).
2. `domain_acc >= 1` in float32 (the `exact` mask).
3. A **neural-free** decoder that reconstructs every pair of \(\mathbb F_p^\times\)
   multiplication. Geometry is not enough (C8).

**Family F.** Phase is a faithful character \(\theta(a)=\phi+2\pi k\log_g(a)/n\)
with \(\gcd(k,n)=1\). The decoder is nearest aligned root of unity: no learned \(W,b\).
On genuine multiplication this program is exact whenever \(\gcd(k,n)=1\); the
scientific claim is that the **exact net** is this character.

**Family Q.** Only for \(p\equiv 3\pmod 4\) and \(\gcd(k,n)=2\): sector identifies
the \(C_n/\{\pm 1\}\) class, shell identifies dlog parity (Legendre). Radius ratio
in every \(\{a,-a\}\) fiber must exceed 1.2, and the sector+shell decoder must
be 100%. Two shells or \(\gcd=2\) alone is not Family Q.

**Also reported, not a family label.** Composition nearest-neighbor
(\(z(a)z(b)\) nearest embedding) and substitution of the ideal character into
the learned readout. Those say whether the embedding, or this net's \(W\),
already implements the law.

In [ ]:
TWO_PI = 2.0 * math.pi
FIBER_RADIUS_RATIO = 1.2


def wrap_np(x):
    return np.arctan2(np.sin(x), np.cos(x))


def dlog_vec(p, dlog_map):
    return np.array([int(dlog_map[a]) for a in range(1, p)], dtype=np.int64)


def family_f_decode(k, scale, phi, p, dlog_map, a, b):
    """Neural-free roots-of-unity classifier. Returns class indices 0..n-1."""
    n = p - 1
    jv = dlog_vec(p, dlog_map).astype(np.float64)
    unit_theta = TWO_PI * int(k) * jv / n
    za = scale * np.stack([np.cos(unit_theta + phi), np.sin(unit_theta + phi)], axis=1)[a - 1]
    zb = scale * np.stack([np.cos(unit_theta + phi), np.sin(unit_theta + phi)], axis=1)[b - 1]
    h = np.stack(
        [za[:, 0] * zb[:, 0] - za[:, 1] * zb[:, 1],
         za[:, 0] * zb[:, 1] + za[:, 1] * zb[:, 0]],
        axis=1,
    )
    doubled = (scale ** 2) * np.stack(
        [np.cos(2.0 * phi + unit_theta), np.sin(2.0 * phi + unit_theta)],
        axis=1,
    )
    return (h @ doubled.T).argmax(axis=1)


def composition_nearest(xy, table):
    """Predict a★b as the embedding nearest to z(a)z(b). Neural-free."""
    z = xy[:, 0] + 1j * xy[:, 1]
    h = z[:, None] * z[None, :]
    dist = np.abs(h[..., None] - z[None, None, :])
    pred = dist.argmin(axis=-1)
    n_ok = int((pred == table).sum())
    return pred, n_ok / table.size, bool(n_ok == table.size)


def product_hidden(xy):
    z = xy[:, 0] + 1j * xy[:, 1]
    h = z[:, None] * z[None, :]
    return np.stack([h.real, h.imag], axis=-1), np.abs(h), np.angle(h)


def n_phases(k, n):
    d = math.gcd(int(k), int(n)) if int(k) else int(n)
    return n // d


def fiber_members(k, p, dlog_map):
    n = p - 1
    q = n_phases(k, n)
    dv = dlog_vec(p, dlog_map)
    out = {}
    for a in range(1, p):
        out.setdefault(int(dv[a - 1] % q), []).append(a)
    return out


def symbolic_decode_q(xy, k, phi, dlog_map, p, rho_even, rho_odd):
    """Sector + same/mixed shell. Returns residues 1..p-1."""
    n = p - 1
    q = n_phases(k, n)
    dv = dlog_vec(p, dlog_map)
    _, pr, pth = product_hidden(xy)
    reps = np.arange(q, dtype=np.float64)
    ideal = 2.0 * phi + TWO_PI * int(k) * reps / n
    delta = wrap_np(pth[..., None] - ideal)
    prod_fiber = np.abs(delta).argmin(axis=-1)
    shells = np.array([rho_even ** 2, rho_even * rho_odd, rho_odd ** 2], dtype=np.float64)
    mixed = np.abs(pr[..., None] - shells).argmin(axis=-1) == 1
    want_even = ~mixed
    pred = np.full(pr.shape, -1, dtype=np.int64)
    for f in range(q):
        members = [a for a in range(1, p) if int(dv[a - 1] % q) == f]
        even_m = [a for a in members if int(dv[a - 1] % 2) == 0]
        odd_m = [a for a in members if int(dv[a - 1] % 2) == 1]
        even_a = even_m[0] if even_m else (members[0] if members else -1)
        odd_a = odd_m[0] if odd_m else (members[-1] if members else -1)
        mask = prod_fiber == f
        pred[mask & want_even] = even_a
        pred[mask & ~want_even] = odd_a
    return pred


def apply_exact_phases(xy, k, phi, dlog_map, p):
    n = p - 1
    r = np.linalg.norm(xy, axis=-1)
    th = phi + TWO_PI * int(k) * dlog_vec(p, dlog_map) / n
    return np.stack([r * np.cos(th), r * np.sin(th)], axis=1)


def apply_two_level_radii(xy, dlog_map, p, rho_even, rho_odd):
    s = dlog_vec(p, dlog_map) % 2
    th = np.arctan2(xy[:, 1], xy[:, 0])
    r = np.where(s == 0, rho_even, rho_odd).astype(np.float64)
    return np.stack([r * np.cos(th), r * np.sin(th)], axis=1)


def crt_product_residue(ja, jb, m, p, g):
    n = p - 1
    u = (ja % m + jb % m) % m
    s = (ja % 2) ^ (jb % 2)
    # j ≡ u (mod m), j ≡ s (mod 2). gcd(m,2)=1 iff p≡3 (mod 4).
    if math.gcd(m, 2) != 1:
        return None
    for j in range(n):
        if j % m == u and j % 2 == s:
            return pow(int(g), int(j) % n, p)
    return None


def certify_family_q(xy, k, phi, p, dlog_map, g):
    """Locked Family Q gates. Decoder reconstructs F_p* multiplication."""
    n = p - 1
    m = n // 2
    dv = dlog_vec(p, dlog_map)
    r = np.linalg.norm(xy, axis=-1)
    th = np.arctan2(xy[:, 1], xy[:, 0])
    aa = np.arange(1, p).repeat(n)
    bb = np.tile(np.arange(1, p), n)
    target = (aa * bb) % p
    out = {"certified": False, "break_at": None, "decoder_acc": None, "decoder_exact": False}

    if math.gcd(int(k), n) != 2:
        out["break_at"] = "gcd_k_not_2"
        return out

    fibers = fiber_members(k, p, dlog_map)
    fiber_ok = all(len(v) == 2 and (v[0] + v[1]) % p == 0 for v in fibers.values())
    sep_ok = []
    large_parity = []
    collapse = []
    for members in fibers.values():
        i, j = members[0] - 1, members[1] - 1
        collapse.append(float(np.abs(wrap_np(th[i] - th[j]))))
        ratio = max(r[i], r[j]) / (min(r[i], r[j]) + 1e-12)
        sep_ok.append(bool(r[i] != r[j] and ratio > FIBER_RADIUS_RATIO))
        large = members[0] if r[i] >= r[j] else members[1]
        large_parity.append(int(dlog_map[large] % 2))

    out["mean_a_minus_a_phase_gap_deg"] = float(np.degrees(np.mean(collapse))) if collapse else float("nan")
    out["fibers_are_pm_a"] = bool(fiber_ok)
    out["frac_fibers_radius_ratio_gt_1_2"] = float(np.mean(sep_ok)) if sep_ok else 0.0
    if large_parity:
        maj = int(np.round(np.mean(large_parity)))
        out["radial_parity_consistency"] = float(np.mean(np.array(large_parity) == maj))
    else:
        out["radial_parity_consistency"] = 0.0

    s = dv % 2
    rho_e = float(r[s == 0].mean())
    rho_o = float(r[s == 1].mean())
    out["rho_even"] = rho_e
    out["rho_odd"] = rho_o

    xy_two = apply_two_level_radii(xy, dlog_map, p, rho_e, rho_o)
    xy_sym = apply_exact_phases(xy_two, k, phi, dlog_map, p)
    dec = symbolic_decode_q(xy_sym, k, phi, dlog_map, p, rho_e, rho_o)
    n_ok = int((dec.reshape(-1) == target).sum())
    n_pairs = n * n
    out["decoder_n_correct"] = n_ok
    out["decoder_acc"] = n_ok / n_pairs
    out["decoder_exact"] = bool(n_ok == n_pairs)

    _, pr, pth = product_hidden(xy_sym)
    shells = np.array([rho_e ** 2, rho_e * rho_o, rho_o ** 2], dtype=np.float64)
    mixed = np.abs(pr[..., None] - shells).argmin(axis=-1) == 1
    tgt = target.reshape(n, n)
    tgt_odd = (dv[tgt - 1] % 2) == 1
    out["shell_xor_acc"] = float((mixed == tgt_odd).mean())

    q_ok = p % 4 == 3
    out["C_n_iso_C_m_x_C2"] = bool(q_ok)
    if q_ok:
        crt = np.empty((n, n), dtype=np.int64)
        for ia, a in enumerate(range(1, p)):
            for ib, b in enumerate(range(1, p)):
                crt[ia, ib] = crt_product_residue(int(dv[ia]), int(dv[ib]), m, p, g)
        out["decoder_equals_crt"] = bool((dec == crt).all())
        out["crt_exact"] = bool((crt == tgt).all())
    else:
        out["decoder_equals_crt"] = False
        out["crt_exact"] = False

    xy_unit = xy / (np.linalg.norm(xy, axis=-1, keepdims=True) + 1e-12)
    xy_u = apply_exact_phases(xy_unit, k, phi, dlog_map, p)
    dec_u = symbolic_decode_q(xy_u, k, phi, dlog_map, p, 1.0, 1.0)
    out["unit_norm_decoder_acc"] = float((dec_u.reshape(-1) == target).mean())

    if not fiber_ok:
        out["break_at"] = "fibers_not_pm_a"
    elif out["frac_fibers_radius_ratio_gt_1_2"] < 1.0 - 1e-12:
        out["break_at"] = "radius_does_not_separate_fiber"
    elif out["radial_parity_consistency"] < 1.0 - 1e-12:
        out["break_at"] = "radius_not_global_dlog_parity"
    elif out["shell_xor_acc"] < 1.0 - 1e-12:
        out["break_at"] = "shell_xor_fails"
    elif not out["decoder_exact"]:
        out["break_at"] = "decoder_not_exact"
    elif not out["C_n_iso_C_m_x_C2"]:
        out["break_at"] = "crt_unavailable_p_eq_1_mod_4"
    elif not out["decoder_equals_crt"]:
        out["break_at"] = "decoder_crt_mismatch"
    else:
        out["certified"] = True
        out["break_at"] = None
    return out


def learned_readout(xy, W_net, b_net):
    """Learned W,b on a (possibly substituted) embedding. Residues 1..p-1."""
    h, _, _ = product_hidden(xy)
    logits = h @ np.asarray(W_net, dtype=np.float64) + np.asarray(b_net, dtype=np.float64)
    return logits.argmax(axis=-1) + 1


if not len(exact_ids):
    print("No exact networks: nothing to certify.")
    cert = pd.DataFrame()
    census = {"population": POP, "n_exact": 0, "n_family_f": 0, "n_family_q": 0, "n_unclassified": 0}
else:
    a_res = a_all_cpu.cpu().numpy().astype(np.int64)
    b_res = b_all_cpu.cpu().numpy().astype(np.int64)
    y_cls = y_all_cpu.cpu().numpy().astype(np.int64)
    table = y_cls.reshape(N, N)
    target_res = (np.arange(1, P)[:, None] * np.arange(1, P)[None, :]) % P
    E_np = E_success.numpy().astype(np.float64)
    W_np = W_success.numpy().astype(np.float64)
    b_np = bias_success.numpy().astype(np.float64)
    domain_ok = (domain_acc[exact_ids] >= 1.0 - 1e-12).cpu().numpy()

    rows = []
    for i, rec in mech3.iterrows():
        li = int(rec["local_idx"])
        xy = E_np[li]
        k = int(rec["best_k"])
        phi = float(rec["phi"])
        scale = float(rec["mean_radius"])
        gcd = int(rec["gcd_k_n"])

        f_pred = family_f_decode(k, scale, phi, P, dlog, a_res, b_res)
        f_acc = float((f_pred == y_cls).mean())
        f_exact = bool((f_pred == y_cls).all())
        family_f = bool(domain_ok[li] and gcd == 1 and f_exact)

        q_cert = certify_family_q(xy, k, phi, P, dlog, g)
        family_q = bool(domain_ok[li] and q_cert["certified"])

        _, c_acc, c_exact = composition_nearest(xy, table)

        z_char = scale * np.stack(
            [np.cos(phi + TWO_PI * k * dlog_vec(P, dlog) / N),
             np.sin(phi + TWO_PI * k * dlog_vec(P, dlog) / N)],
            axis=1,
        )
        sub = learned_readout(z_char, W_np[li], b_np[li])
        sub_acc = float((sub == target_res).mean())
        sub_exact = bool((sub == target_res).all())

        if family_f:
            family = "FAMILY_F"
        elif family_q:
            family = "FAMILY_Q"
        else:
            family = "EXACT-UNCLASSIFIED"

        rows.append({
            "net": int(rec["net"]),
            "domain_exact": bool(domain_ok[li]),
            "best_k": k,
            "gcd_k_n": gcd,
            "phase_mae_deg": float(rec["phase_mae_deg"]),
            "mean_abs_deg": float(rec["mean_abs_deg"]),
            "q_odd_over_even": float(rec["q_odd_over_even"]),
            "radius_parity_separation": float(rec["radius_parity_separation"]),
            "f_decoder_acc": f_acc,
            "f_decoder_exact": f_exact,
            "q_decoder_acc": q_cert.get("decoder_acc"),
            "q_decoder_exact": q_cert.get("decoder_exact"),
            "q_certified": q_cert["certified"],
            "q_break_at": q_cert["break_at"],
            "composition_acc": c_acc,
            "composition_exact": c_exact,
            "substitution_acc": sub_acc,
            "substitution_exact": sub_exact,
            "family": family,
        })

    cert = pd.DataFrame(rows)
    n_f = int((cert["family"] == "FAMILY_F").sum())
    n_q = int((cert["family"] == "FAMILY_Q").sum())
    n_u = int((cert["family"] == "EXACT-UNCLASSIFIED").sum())
    census = {
        "p": P,
        "population": POP,
        "seed": SEED,
        "split": SPLIT,
        "n_exact": int(len(exact_ids)),
        "n_family_f": n_f,
        "n_family_q": n_q,
        "n_unclassified": n_u,
        "n_composition_exact": int(cert["composition_exact"].sum()),
        "n_substitution_exact": int(cert["substitution_exact"].sum()),
        "frac_exact": int(len(exact_ids)) / POP,
        "frac_f_among_exact": n_f / max(len(exact_ids), 1),
        "frac_q_among_exact": n_q / max(len(exact_ids), 1),
    }
    (RUN_DIR / "certificate_census.json").write_text(json.dumps(census, indent=2))
    cert.to_csv(RUN_DIR / "certificates.csv", index=False)
    print(json.dumps(census, indent=2))
    print("\nFamily counts:", cert["family"].value_counts().to_dict())
    print("\nQ break reasons among gcd=2:")
    gcd2 = cert[cert["gcd_k_n"] == 2]
    if len(gcd2):
        print(gcd2["q_break_at"].value_counts(dropna=False).to_string())
    else:
        print("none")
    display(cert.sort_values(["family", "phase_mae_deg"]).head(30))

In [ ]:
if len(exact_ids) and len(cert):
    geo = mech3.merge(
        cert[["net", "domain_exact", "f_decoder_exact", "q_certified",
              "q_break_at", "composition_exact", "substitution_exact", "family"]],
        on="net",
    )
    geo.to_csv(RUN_DIR / "mechanistic_with_certificates.csv", index=False)

    print("Geometry is not a certificate. Family is.")
    print("All gcd=1 exact nets Family F:",
          bool(((geo["gcd_k_n"] == 1) == (geo["family"] == "FAMILY_F")).all()))
    print("Any Family Q with q far from 1 and uncertified?",
          int(((geo["family"] != "FAMILY_Q") & (geo["gcd_k_n"] == 2)).sum()),
          "gcd=2 exact nets that failed Q")

    cols = [
        "net", "best_k", "gcd_k_n", "phase_mae_deg", "radius_cv", "mean_radius",
        "phi", "mean_abs_deg", "max_abs_deg", "gauge_deg",
        "rho_even", "rho_odd", "q_odd_over_even", "radius_parity_separation",
        "f_decoder_exact", "q_certified", "composition_exact", "family",
    ]
    top_f = geo.sort_values(["gcd_k_n", "phase_mae_deg"]).head(30)
    display(top_f[cols])

    print("\nUnclassified exact nets (geometry without a decoder):")
    u = geo[geo["family"] == "EXACT-UNCLASSIFIED"]
    if len(u):
        display(u[cols].head(20))
    else:
        print("none")
else:
    print("No certificates to merge.")


# GPU utilization after the run

Within a live Jupyter session, `nvidia-smi` gives an instantaneous view. OSC also provides
post-job GPU accounting tools such as `gpu-seff` / `osc-seff` when you know the Slurm job ID.

The next cell prints the Slurm environment and current GPU state.


In [ ]:

print("SLURM_JOB_ID:", os.environ.get("SLURM_JOB_ID"))
print("SLURM_JOB_NAME:", os.environ.get("SLURM_JOB_NAME"))
print("SLURM_JOB_NODELIST:", os.environ.get("SLURM_JOB_NODELIST"))

subprocess.run(["nvidia-smi"], check=False)

print("\nIf gpu-seff is available, after the job/session completes you can run:")
if os.environ.get("SLURM_JOB_ID"):
    print(f"gpu-seff -v {os.environ['SLURM_JOB_ID']}")
    print(f"osc-seff {os.environ['SLURM_JOB_ID']}")
else:
    print("gpu-seff -v <jobid>")
    print("osc-seff <jobid>")



# Suggested OSC scaling experiments

Once the notebook works unchanged, useful tests are:

1. Run `POP = 1024, 4096, 16384` and compare network-steps/s.
2. Compare one OSC GPU against your local RTX 5060 Ti using the benchmark cell.
3. Keep float32 for the scientific experiment unless precision is explicitly being tested.
4. Record exact-solver frequency **and** Family F / Family Q counts separately
   for every hardware/precision condition. A larger population that only tightens
   phase MAE has not closed the loop.
5. If using multiple GPUs later, split independent population members across GPUs rather than
   synchronizing individual tiny networks. The experiment is embarrassingly parallel at the
   population level.

A Cardinal H100 or Ascend A100 has far more VRAM than this experiment normally requires, so the
most useful OSC advantage may be the ability to run **much larger populations and independent
replicates simultaneously**, not merely making one 1024-network population faster.

Outputs written under `RUN_DIR`:

- `population.pt` — full `E`, `W`, `bias`, `domain_acc`, `exact` (ids are slots)
- `successful.pt` — exact subset plus `network_ids`
- `mechanistic_summary.csv`, `mechanistic_with_radius.csv`
- `certificates.csv`, `certificate_census.json`, `mechanistic_with_certificates.csv`
